In [ ]:
import pvporcupine
import pyaudio
import struct

def create_wake_word_detector():
    # 创建Porcupine实例，使用预定义的中文唤醒词
    porcupine = pvporcupine.create(
        keywords=['hey google', 'ok google']  # 可以替换成支持的中文关键词
    )
    return porcupine

def detect_wake_word():
    porcupine = create_wake_word_detector()
    
    # 初始化音频流
    pa = pyaudio.PyAudio()
    audio_stream = pa.open(
        rate=porcupine.sample_rate,
        channels=1,
        format=pyaudio.paInt16,
        input=True,
        frames_per_buffer=porcupine.frame_length
    )
    
    print("等待唤醒词...")
    
    try:
        while True:
            # 读取音频数据
            pcm = audio_stream.read(porcupine.frame_length)
            pcm = struct.unpack_from("h" * porcupine.frame_length, pcm)
            
            # 检测唤醒词
            keyword_index = porcupine.process(pcm)
            
            if keyword_index >= 0:
                print(f"检测到唤醒词: {porcupine.keyword_names[keyword_index]}")
                # 在这里触发后续操作
                
    except KeyboardInterrupt:
        print("停止监听...")
    finally:
        audio_stream.close()
        pa.terminate()
        porcupine.delete()